In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import glob

In [2]:
# parquet dosyalarını sıralı şekilde alıyorum
files = sorted(glob.glob("../data/*.parquet"))

print(files)

['../data/yellow_tripdata_2015-01.parquet', '../data/yellow_tripdata_2015-02.parquet', '../data/yellow_tripdata_2015-03.parquet', '../data/yellow_tripdata_2015-04.parquet', '../data/yellow_tripdata_2015-05.parquet', '../data/yellow_tripdata_2015-06.parquet', '../data/yellow_tripdata_2015-07.parquet', '../data/yellow_tripdata_2015-08.parquet', '../data/yellow_tripdata_2015-09.parquet', '../data/yellow_tripdata_2015-10.parquet', '../data/yellow_tripdata_2015-11.parquet', '../data/yellow_tripdata_2015-12.parquet']


In [3]:
# İlk olarak sadece Ocak ayı verisini inceliyorum
df_january = pd.read_parquet("../data/yellow_tripdata_2015-01.parquet")
df_january.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2015-01-01 00:11:33,2015-01-01 00:16:48,1,1.0,1,N,41,166,1,5.7,0.5,0.5,1.40,0.0,0.0,8.40,None,None
1,1,2015-01-01 00:18:24,2015-01-01 00:24:20,1,0.9,1,N,166,238,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,None,None
2,1,2015-01-01 00:26:19,2015-01-01 00:41:06,1,3.5,1,N,238,162,1,13.2,0.5,0.5,2.90,0.0,0.0,17.40,None,None
3,1,2015-01-01 00:45:26,2015-01-01 00:53:20,1,2.1,1,N,162,263,1,8.2,0.5,0.5,2.37,0.0,0.0,11.87,None,None
4,1,2015-01-01 00:59:21,2015-01-01 01:05:24,1,1.0,1,N,236,141,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,None,None


# Veriye genel bakış


In [4]:
# satır ve sütun sayısının kontrol edilmesi
df_january.shape

(12741035, 19)

In [5]:
# sütun isimlerini kontrol ediyorum
df_january.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee'],
      dtype='str')

In [6]:
# veri tiplerinin kontrol edilmesi
df_january.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                   int64
trip_distance                   float64
RatecodeID                        int64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge             object
airport_fee                      object
dtype: object

In [7]:
# Eksik değerleri kontrol ediyorum
df_january.isnull().sum()

VendorID                        0
tpep_pickup_datetime            0
tpep_dropoff_datetime           0
passenger_count                 0
trip_distance                   0
RatecodeID                      0
store_and_fwd_flag              0
PULocationID                    0
DOLocationID                    0
payment_type                    0
fare_amount                     0
extra                           0
mta_tax                         0
tip_amount                      0
tolls_amount                    0
improvement_surcharge           3
total_amount                    0
congestion_surcharge     12741035
airport_fee              12741035
dtype: int64

In [8]:
# pickup sütununu datetime türüne dönüştürüyorum
df_january["tpep_pickup_datetime"] = pd.to_datetime(
    df_january["tpep_pickup_datetime"],
    errors="coerce"
)
# Ocak ayı için saatlik trip sayısını hesaplıyorum
january_hourly = (
    df_january.set_index("tpep_pickup_datetime")
    .resample("h")
    .size()
    .reset_index(name="trip_count")
)
january_hourly.head()

,tpep_pickup_datetime,trip_count
0,2015-01-01 00:00:00,28312
1,2015-01-01 01:00:00,31707
2,2015-01-01 02:00:00,28068
3,2015-01-01 03:00:00,24288
4,2015-01-01 04:00:00,17081


### Aylık dosyaların eklenmesi


In [ ]:
# İlk 6 ay için gerekli sütunları okuyorum
all_data = []

for file in files[:6]:

    print("Reading:", file)

    temp_df = pd.read_parquet(
        file,
        columns=[
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "passenger_count",
            "trip_distance",
            "RatecodeID",
            "payment_type",
            "fare_amount",
            "extra",
            "mta_tax",
            "tip_amount",
            "tolls_amount",
            "total_amount",
            "improvement_surcharge",
            "airport_fee",
            "congestion_surcharge"
        ]
    )
    all_data.append(temp_df)
# ilk 6 ayı tek dataframe içinde birleştiriyorum
df = pd.concat(all_data, ignore_index=True)

print("Dataset shape:", df.shape)

Reading: ../data/yellow_tripdata_2015-01.parquet
Reading: ../data/yellow_tripdata_2015-02.parquet
Reading: ../data/yellow_tripdata_2015-03.parquet
Reading: ../data/yellow_tripdata_2015-04.parquet
Reading: ../data/yellow_tripdata_2015-05.parquet
Reading: ../data/yellow_tripdata_2015-06.parquet
Dataset shape: (77072751, 15)


In [21]:
# Sayısal sütunların temel istatistiklerini inceliyorum
df.describe()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount,improvement_surcharge
count,77072751,77072751,7.707275e+07,7.707275e+07,7.707275e+07,7.707275e+07,7.707275e+07,7.707275e+07,7.707275e+07,7.707275e+07,7.707275e+07,7.707275e+07,7.707275e+07
mean,2015-04-01 06:30:35.084935,2015-04-01 06:55:19.993336,1.677936e+00,1.641442e+01,1.039064e+00,1.378112e+00,1.272111e+01,3.144517e-01,4.977200e-01,1.728755e+00,2.892661e-01,1.586908e+01,2.970713e-01
min,2015-01-01 00:00:00,1973-05-09 09:17:59,0.000000e+00,-4.084012e+07,1.000000e+00,1.000000e+00,-4.960000e+02,-7.900000e+01,-5.000000e-01,-1.180000e+02,-9.400000e+01,-4.963000e+02,-3.000000e-01
25%,2015-02-15 21:01:52,2015-02-15 21:15:06.500000,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,6.500000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,8.380000e+00,3.000000e-01
50%,2015-04-01 00:45:29,2015-04-01 01:01:30,1.000000e+00,1.700000e+00,1.000000e+00,1.000000e+00,9.500000e+00,0.000000e+00,5.000000e-01,1.150000e+00,0.000000e+00,1.180000e+01,3.000000e-01
75%,2015-05-15 07:33:11,2015-05-15 07:47:30,2.000000e+00,3.180000e+00,1.000000e+00,2.000000e+00,1.450000e+01,5.000000e-01,5.000000e-01,2.260000e+00,0.000000e+00,1.775000e+01,3.000000e-01
max,2015-06-30 23:59:59,2253-08-23 07:56:38,9.000000e+00,5.901661e+07,9.900000e+01,5.000000e+00,5.033255e+05,6.524200e+02,8.080000e+01,3.950589e+06,1.450090e+03,3.950612e+06,7.000000e-01
std,NaN,NaN,1.335309e+00,1.411705e+04,6.041210e-01,4.981493e-01,8.723489e+01,4.636048e-01,4.662914e-02,4.500058e+02,1.551062e+00,4.687409e+02,2.995009e-02


In [35]:
# sayısal sütunların min, max, mean ve median değerlerini kontrol ediyorum
numeric_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
]
for col in numeric_columns:

    print("\nColumn:", col)

    print("Minimum:", df[col].min())

    print("Maximum:", df[col].max())

    print("Mean:", df[col].mean())

    print("Median:", df[col].median())


Column: passenger_count
Minimum: 0
Maximum: 9
Mean: 1.6779355520863657
Median: 1.0

Column: trip_distance
Minimum: -40840124.4
Maximum: 59016609.3
Mean: 16.41441570289868
Median: 1.7

Column: fare_amount
Minimum: -496.0
Maximum: 503325.53
Mean: 12.721113349645453
Median: 9.5

Column: tip_amount
Minimum: -118.0
Maximum: 3950588.8
Mean: 1.7287553704940413
Median: 1.15

Column: tolls_amount
Minimum: -94.0
Maximum: 1450.09
Mean: 0.2892661377819509
Median: 0.0

Column: total_amount
Minimum: -496.3
Maximum: 3950611.6
Mean: 15.869080094468147
Median: 11.8


In [31]:
# # negatif ve mantıksız değerleri kontrol ettikten sonra outlier temizleme işlemlerinde IQR yöntemini kullandım
print("Passenger count <= 0:", (df["passenger_count"] <= 0).sum())

print("Trip distance <= 0:", (df["trip_distance"] <= 0).sum())

print("Fare amount <= 0:", (df["fare_amount"] <= 0).sum())

print("Negative tip amounts:", (df["tip_amount"] < 0).sum())



Passenger count <= 0: 36366
Trip distance <= 0: 458002
Fare amount <= 0: 43487
Negative tip amounts: 488


In [32]:
# trip_distance için IQR hesaplıyorum
Q1_distance = df["trip_distance"].quantile(0.25)

Q3_distance = df["trip_distance"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance

upper_distance = Q3_distance + 1.5 * IQR_distance


# fare_amount için IQR hesaplıyorum
Q1_fare = df["fare_amount"].quantile(0.25)

Q3_fare = df["fare_amount"].quantile(0.75)

IQR_fare = Q3_fare - Q1_fare

lower_fare = Q1_fare - 1.5 * IQR_fare

upper_fare = Q3_fare + 1.5 * IQR_fare


print("Trip distance upper bound:", upper_distance)

print("Fare amount upper bound:", upper_fare)

Trip distance upper bound: 6.450000000000001
Fare amount upper bound: 26.5


## Veri dağılımı normal olmadığı ve veride çok fazla aykırı değer bulunduğu için outlier temizleme işleminde IQR yöntemini kullandım

In [25]:
# IQR yöntemine göre mantıksız değerleri temizliyorum
condition = (
    (df["passenger_count"] > 0) &
    (df["passenger_count"] <= 6) &
    (df["trip_distance"] >= lower_distance) &
    (df["trip_distance"] <= upper_distance) &
    (df["fare_amount"] >= lower_fare) &
    (df["fare_amount"] <= upper_fare) &
    (df["tip_amount"] >= 0)
)

df_clean = df.loc[condition].copy()

print("Before cleaning:", df.shape)
print("After cleaning:", df_clean.shape)

Before cleaning: (77072751, 15)
After cleaning: (68870406, 15)


In [26]:
# # pickup datetime sütununu datetime formatına dönüştürülmesi
df_clean["tpep_pickup_datetime"] = pd.to_datetime(
    df_clean["tpep_pickup_datetime"],
    errors="coerce"
)
# dropoff datetime sütununu datetime formatına dönüştürüyorum
df_clean["tpep_dropoff_datetime"] = pd.to_datetime(
    df_clean["tpep_dropoff_datetime"],
    errors="coerce"
)
# boş datetime değerlerini kaldırıyorum
df_clean = df_clean.dropna(
    subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime"
    ]
)
# sadece 2015 yılı içindeki verileri tutuyorum
df_clean = df_clean[
    (df_clean["tpep_pickup_datetime"] >= "2015-01-01") &
    (df_clean["tpep_pickup_datetime"] < "2016-01-01")
]
print("Cleaned dataset shape:", df_clean.shape)

Cleaned dataset shape: (68870406, 15)


In [27]:
# temizlik sonrası temel istatistikleri tekrar kontrol ediyorum
df_clean.describe()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount,improvement_surcharge
count,68870406,68870406,6.887041e+07,6.887041e+07,6.887041e+07,6.887041e+07,6.887041e+07,6.887041e+07,6.887041e+07,6.887041e+07,6.887041e+07,6.887041e+07,6.887040e+07
mean,2015-03-31 20:23:11.201435,2015-03-31 20:44:53.774332,1.674677e+00,1.927519e+00,1.006464e+00,1.385522e+00,9.811969e+00,3.200276e-01,4.992084e-01,1.340002e+00,1.768852e-02,1.230862e+01,2.970959e-01
min,2015-01-01 00:00:00,1973-05-09 09:17:59,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,-5.500000e+00,-7.900000e+01,-5.000000e-01,0.000000e+00,-9.400000e+01,-1.048800e+02,-3.000000e-01
25%,2015-02-15 11:37:22,2015-02-15 11:47:29,1.000000e+00,9.600000e-01,1.000000e+00,1.000000e+00,6.500000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,8.300000e+00,3.000000e-01
50%,2015-03-31 12:40:31,2015-03-31 12:55:27,1.000000e+00,1.550000e+00,1.000000e+00,1.000000e+00,8.500000e+00,0.000000e+00,5.000000e-01,1.000000e+00,0.000000e+00,1.100000e+01,3.000000e-01
75%,2015-05-14 19:32:49,2015-05-14 19:47:13,2.000000e+00,2.530000e+00,1.000000e+00,2.000000e+00,1.250000e+01,5.000000e-01,5.000000e-01,2.050000e+00,0.000000e+00,1.530000e+01,3.000000e-01
max,2015-06-30 23:59:59,2253-08-23 07:56:38,6.000000e+00,6.450000e+00,9.900000e+01,5.000000e+00,2.650000e+01,6.524200e+02,8.080000e+01,3.950589e+06,9.489000e+02,3.950612e+06,7.000000e-01
std,NaN,NaN,1.334151e+00,1.318752e+00,4.611096e-01,4.990575e-01,4.616385e+00,4.299729e-01,3.411477e-02,4.760447e+02,5.433618e-01,4.871695e+02,2.978257e-02


### Yıl boyu saatlik trip sayısı

In [28]:
# temizlenmiş veri ile saatlik trip sayısını hesaplıyorum
hourly_data = (
    df_clean.set_index("tpep_pickup_datetime")
    .resample("h")
    .size()
    .reset_index(name="trip_count")
)
hourly_data.head()

,tpep_pickup_datetime,trip_count
0,2015-01-01 00:00:00,25574
1,2015-01-01 01:00:00,28160
2,2015-01-01 02:00:00,24452
3,2015-01-01 03:00:00,20713
4,2015-01-01 04:00:00,14171


In [37]:
hourly_data.describe()

,tpep_pickup_datetime,trip_count
count,4344,4344.000000
mean,2015-04-01 11:30:00,15854.145028
min,2015-01-01 00:00:00,11.000000
25%,2015-02-15 05:45:00,10377.000000
50%,2015-04-01 11:30:00,17850.500000
75%,2015-05-16 17:15:00,20935.250000
max,2015-06-30 23:00:00,30976.000000
std,NaN,7349.491408


In [ ]:
hourly_data.shape

(4344, 2)

In [30]:
# saatlik trip count verisini csv olarak kaydediyorum
hourly_data.to_csv(
    "../outputs/hourly_trip_counts_2015.csv",
    index=False
)
print("CSV dosyası kaydedildi")

CSV dosyası kaydedildi
